In [10]:
import pandas as pd


df = pd.read_excel('sira_fatura_kontrol.xlsx', sheet_name='Tedarikci_Yil_Ozet', header=2)

# Kolon isimlerinin başında/sonunda gizli boşluk varsa temizleme
df.columns = df.columns.str.strip()

# --- TEMİZLİK FONKSİYONLARI ---

def clean_currency(value):
    if isinstance(value, str):
        value = value.replace('.', '').replace(',', '.')
    try:
        return float(value)
    except:
        return value

def clean_percent(value):
    if isinstance(value, str):
        value = value.replace('%', '').replace(',', '.')
        try:
            return float(value) / 100
        except:
            return value
    return value

# --- colmun-based transformations ---

# Para birimi ve sayısal tutarları temizleme
currency_cols = ['Toplam_Tutar', 'Ardisik_Seriye_Dahil_Tutar']
for col in currency_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_currency)

# Yüzdelik oranları temizleme
percent_cols = ['Ardisik_Oran_Adet', 'Ardisik_Oran_Tutar']
for col in percent_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_percent)

# Sayısal olması gereken diğer adet kolonları da, in case,
int_cols = ['Toplam_Fatura_Adedi', 'Ardisik_Seri_Sayisi', 'Ardisik_Seriye_Dahil_Fatura_Adedi', 'Prefix_Sayisi']
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Boş (NaN) hücreleri 0
df = df.fillna(0)

print("Excel başlık kayması düzeltildi ve temizlik başarıyla tamamlandı!")
df.head()

Excel başlık kayması düzeltildi ve temizlik başarıyla tamamlandı!


,Yıl,Tedarikçi,Toplam_Fatura_Adedi,Toplam_Tutar,Ardisik_Seri_Sayisi,Ardisik_Seriye_Dahil_Fatura_Adedi,Ardisik_Seriye_Dahil_Tutar,Ardisik_Oran_Adet,Ardisik_Oran_Tutar,Prefix_Sayisi
0,2023,. EKİNOKS MAKİNE MÜH. VE END. EKP. SAN.TİC.LTD...,4,25843.18,2,4,25843.18,1.000000,1.000000,1
1,2023,(Boş),83,349896.10,5,13,82724.65,0.156627,0.236426,25
2,2023,ABECE GRUP ÇEVRE VE İŞ GÜVENLİĞİ MÜHENDİSLİK H...,12,47988.24,0,0,0.00,0.000000,0.000000,2
3,2023,Acıbadem Sağlık Hizmetleri ve Tic.A.Ş.,1,3425.93,0,0,0.00,0.000000,0.000000,1
4,2023,ACTECON DANIŞMANLIK A.Ş.,1,5000.00,0,0,0.00,0.000000,0.000000,1


In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest

# 1. Modele sokacağımız featurlar
# Tedarikçi ismi ve Yıl gibi metinsel alanlar model hesaplamasına dahil edilmiyor
features = [
    'Toplam_Fatura_Adedi', 'Toplam_Tutar', 'Ardisik_Seri_Sayisi', 
    'Ardisik_Seriye_Dahil_Fatura_Adedi', 'Ardisik_Seriye_Dahil_Tutar', 
    'Ardisik_Oran_Adet', 'Ardisik_Oran_Tutar', 'Prefix_Sayisi'
]

X = df[features]

# 2. scaling the data (StandardScaler: ortalamayı 0, standart sapmayı 1 yapar)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- CLUSTERING (GRUPLAMA) ---
# Tedarikçileri benzer davranışlarına göre 4 farklı kümeye ayıralım
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Kume_No'] = kmeans.fit_transform(X_scaled).argmin(axis=1) # Her tedarikçiye bir küme numarası verdik

# --- ANOMALY DETECTION  ---
# isolation Forest algoritması ile şüpheli/aykırı tedarikçileri buluyoruz
# contamination=0.05 verinin %5'ini anomali (şüpheli) olarak işaretliyoruz
iso_forest = IsolationForest(contamination=0.05, random_state=42)
# Algoritma normal veriye 1, anomaliye -1 değerini verir
df['Anomali_Durumu'] = iso_forest.fit_predict(X_scaled)
df['Anomali_Durumu'] = df['Anomali_Durumu'].map({1: 'Normal', -1: 'ŞÜPHELİ (ANOMALİ)'})

print("Modelleme tamamlandı! Sonuçlar tabloya eklendi.")


Modelleme tamamlandı! Sonuçlar tabloya eklendi.


In [12]:
# 1. Sonuçları içeren yeni excel dosyaysı
output_file = 'sira_fatura_kontrol_sonuclar.xlsx'
df.to_excel(output_file, index=False)
print(f"1. ADIM BAŞARILI: Tüm sonuçlar '{output_file}' adıyla klasörüne kaydedildi!\n")

# 2. Küme summary
# Her kümenin ortalama değerleri
print("--- KÜME ÖZETLERİ (Hangi küme ne anlama geliyor?) ---")
kume_ozet = df.groupby('Kume_No')[['Toplam_Tutar', 'Toplam_Fatura_Adedi', 'Ardisik_Oran_Tutar', 'Prefix_Sayisi']].mean()
print(kume_ozet)

# 3. Kaç tane şüpheli/anomali tedarikçi bulduk?
print("\n--- ANOMALİ DAĞILIMI ---")
print(df['Anomali_Durumu'].value_counts())

1. ADIM BAŞARILI: Tüm sonuçlar 'sira_fatura_kontrol_sonuclar.xlsx' adıyla klasörüne kaydedildi!

--- KÜME ÖZETLERİ (Hangi küme ne anlama geliyor?) ---
         Toplam_Tutar  Toplam_Fatura_Adedi  Ardisik_Oran_Tutar  Prefix_Sayisi
Kume_No                                                                      
0        2.624538e+05              7.94987            0.014589       1.126953
1        8.634209e+05             27.25000            0.690028       1.628788
2        2.056522e+06            874.00000            0.633764       1.000000
3        2.381497e+06            226.50000            0.599039       6.562500
4        6.717289e+07              2.00000            1.000000       1.000000

--- ANOMALİ DAĞILIMI ---
Anomali_Durumu
Normal               1602
ŞÜPHELİ (ANOMALİ)      85
Name: count, dtype: int64
